In [1]:
import pandas as pd
import datetime
import sqlite3
import pymysql
import pandas.io.sql as psql
from datetime import datetime as dt
import numpy as np
import pandas.tseries.offsets as offsets
import sqlalchemy as sqa
import matplotlib.pyplot as plt
import python_ss as ps
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import json

import os
import ast
import db_dtypes
from google.cloud import bigquery
from google.oauth2 import service_account
from google.cloud import secretmanager

In [2]:
def access_secret_version(project_id, secret_id, version_id='latest'):
    client = secretmanager.SecretManagerServiceClient()

    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    response = client.access_secret_version(request={"name": name})
    payload = response.payload.data.decode("UTF-8")
    return ast.literal_eval(payload)

In [3]:
# 上記関数を実行するコードが記載されています。こちらもそのままお使いください。
credentials = service_account.Credentials.from_service_account_info(
access_secret_version('temp-for-sandbox', 'TEMP_CREDENTIAL_KEY'),
scopes=["https://www.googleapis.com/auth/cloud-platform"],)


C:\Users\suehara\Anaconda3\lib\site-packages\google\auth\_default.py:78: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [4]:
dt_now = datetime.datetime.now()
dt_now = dt_now.strftime('%Y%m%d')
dt_now

'20230531'

In [5]:
##extitle + dt_now + ".xlsx"
path1 = "//172.16.0.232/CoffeeCrazy/経営ソリューション事業部/□スカウト推進部□/060 ※社外秘※共通シート/アンタッチャブルリスト/"
extitle = "【発注用】アンタッチャ一覧"
extitle = extitle + dt_now + ".xlsx"
extitle = path1 + extitle
extitle

'//172.16.0.232/CoffeeCrazy/経営ソリューション事業部/□スカウト推進部□/060 ※社外秘※共通シート/アンタッチャブルリスト/【発注用】アンタッチャ一覧20230531.xlsx'

In [6]:
df = pd.read_excel(extitle, sheet_name = 'アンタッチャ一覧',usecols=[3,6,14,16])

In [26]:
untouch = df.copy()

In [27]:
untouch

,CompanyName,JobName,BossName,Note
0,CELVINVIETNAMCOMPANYLIMITED.,NaN,NaN,2012/04/18アンタッチャ。依頼：市場開発部。株式会社ｾﾙｳﾞｧﾝの関連会社。期間：最...
1,株式会社ｻｰｸﾙ,不動産,NaN,988116552とは別会社 アンタッチャ担当：藤田 依頼：スカウト推進部
2,東武ｴｽﾃｰﾄ,NaN,NaN,期間：無制限? クライアント
3,株式会社ｵｰｼﾞｯｸ(ｵｰｼﾞｯｸｸﾞﾙｰﾌﾟ),動力伝導装置製造,田中 文彦,21/5/7ｱﾝﾀ。依頼：市開。期間：最終入金から1年間。20/7/28ｱﾝﾀ。依頼：顧問名...
4,株式会社ﾋﾞｰｽﾞｸﾘｴｲﾃｨﾌﾞｽﾞ,広告代理業,NaN,07/7/26アンタッチャ 依頼：市場開発 株式会社ﾋﾞｰｽﾞｲﾝﾀｰﾅｼｮﾅﾙの関連会社
...,...,...,...,...
26033,株式会社ﾊﾛ,ソフト受託開発,矢野 卓,2021/02/12アンタッチャ。依頼：顧問名鑑事業部。期間：最終入金から1年間。2012/...
26034,株式会社ﾎﾜｲﾄﾋﾞｼﾞﾈｽｲﾆｼｱﾃｨﾌﾞ,産業用電気機器卸,谷井 剛,2013/06/20アンタッチャ。依頼：市場開発部。株式会社ﾌｫｰﾊﾞﾙの関連会社。期間：最...
26035,株式会社ﾊﾟﾜｰﾌｨﾅﾝｼｬﾙｸﾞﾙｰﾌﾟ,NaN,NaN,2010/03/09アンタッチャ。依頼：市場開発部。株式会社ﾊﾟﾜｰｺﾝｻﾙﾃｨﾝｸﾞﾈｯﾄ...
26036,ﾊﾟﾅﾌｰｽﾞ株式会社,他の水産食料品製造,NaN,2021/10/14 倒産。 https://n-seikei.jp/2021/07/pos...


In [29]:
path2 = "//172.16.0.232/CoffeeCrazy3/新規事業室（藤社長）/転機_候補者対応関連/登録者エクセル格納フォルダ/相馬/アンタッチャ一覧.xlsx"
untouch.to_excel(path2,sheet_name = 'アンタッチャ一覧',index=False)
path2

'//172.16.0.232/CoffeeCrazy3/新規事業室（藤社長）/転機_候補者対応関連/登録者エクセル格納フォルダ/相馬/アンタッチャ一覧.xlsx'

In [30]:
tenki_candidate_query = """
SELECT
id as tenki_ID,
concat(replace(shi," ",""),replace(mei," ","")) as name,
replace(replace(replace(comp_name," ",""),"　",""),"・","") as CompanyName
FROM `temp-380708.live_tenki.user_info`
where replace(shi," ","") != ""
AND replace(mei," ","") != ""
AND id >= 49915 
order by id
"""
##created_at >= CURRENT_DATE('Asia/Tokyo')-10

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
tenki_candidate = client.query(tenki_candidate_query).result().to_dataframe()

In [31]:
tenki_candidate

,tenki_ID,name,CompanyName
0,49916,宮島信文,株式会社ジーテイスト
1,49917,藤川卓也,クレディスイス証券•銀行
2,49918,南部正明,NoveWorks株式会社
3,49919,小野満,中日販売株式会社
4,49920,富田輝明,株式会社清光社
...,...,...,...
44443,105337,髙橋一晃,株式会社TBSテレビ
44444,105338,向井昌博,ネクスティシステムデザイン株式会社
44445,105340,成山光史,株式会社フェース
44446,105341,森勝彦,株式会社Ｇファクトリー


In [32]:
untouch

,CompanyName,JobName,BossName,Note
0,CELVINVIETNAMCOMPANYLIMITED.,NaN,NaN,2012/04/18アンタッチャ。依頼：市場開発部。株式会社ｾﾙｳﾞｧﾝの関連会社。期間：最...
1,株式会社ｻｰｸﾙ,不動産,NaN,988116552とは別会社 アンタッチャ担当：藤田 依頼：スカウト推進部
2,東武ｴｽﾃｰﾄ,NaN,NaN,期間：無制限? クライアント
3,株式会社ｵｰｼﾞｯｸ(ｵｰｼﾞｯｸｸﾞﾙｰﾌﾟ),動力伝導装置製造,田中 文彦,21/5/7ｱﾝﾀ。依頼：市開。期間：最終入金から1年間。20/7/28ｱﾝﾀ。依頼：顧問名...
4,株式会社ﾋﾞｰｽﾞｸﾘｴｲﾃｨﾌﾞｽﾞ,広告代理業,NaN,07/7/26アンタッチャ 依頼：市場開発 株式会社ﾋﾞｰｽﾞｲﾝﾀｰﾅｼｮﾅﾙの関連会社
...,...,...,...,...
26033,株式会社ﾊﾛ,ソフト受託開発,矢野 卓,2021/02/12アンタッチャ。依頼：顧問名鑑事業部。期間：最終入金から1年間。2012/...
26034,株式会社ﾎﾜｲﾄﾋﾞｼﾞﾈｽｲﾆｼｱﾃｨﾌﾞ,産業用電気機器卸,谷井 剛,2013/06/20アンタッチャ。依頼：市場開発部。株式会社ﾌｫｰﾊﾞﾙの関連会社。期間：最...
26035,株式会社ﾊﾟﾜｰﾌｨﾅﾝｼｬﾙｸﾞﾙｰﾌﾟ,NaN,NaN,2010/03/09アンタッチャ。依頼：市場開発部。株式会社ﾊﾟﾜｰｺﾝｻﾙﾃｨﾝｸﾞﾈｯﾄ...
26036,ﾊﾟﾅﾌｰｽﾞ株式会社,他の水産食料品製造,NaN,2021/10/14 倒産。 https://n-seikei.jp/2021/07/pos...


In [33]:
# In[5]:

untauch = untouch
untauch['CompanyName'] = untauch['CompanyName'].replace(" ","").replace("　","").replace("・","")
untauch = untauch[['CompanyName']]
untauch_list = untauch['CompanyName'].tolist()
untauch

,CompanyName
0,CELVINVIETNAMCOMPANYLIMITED.
1,株式会社ｻｰｸﾙ
2,東武ｴｽﾃｰﾄ
3,株式会社ｵｰｼﾞｯｸ(ｵｰｼﾞｯｸｸﾞﾙｰﾌﾟ)
4,株式会社ﾋﾞｰｽﾞｸﾘｴｲﾃｨﾌﾞｽﾞ
...,...
26033,株式会社ﾊﾛ
26034,株式会社ﾎﾜｲﾄﾋﾞｼﾞﾈｽｲﾆｼｱﾃｨﾌﾞ
26035,株式会社ﾊﾟﾜｰﾌｨﾅﾝｼｬﾙｸﾞﾙｰﾌﾟ
26036,ﾊﾟﾅﾌｰｽﾞ株式会社


In [34]:
import pandas as pd
import pymysql
import jaconv
import pickle
import os.path
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request

In [35]:
def get_untauch(x):
    count = 0
    for i in untauch_list:
        if x in i:
            count += 1
    if count > 0:
        return '現S同意書回収'
    else:
        return None

def remove_kabu(x):
    remove_dict = {"株式会社": "", "（株）": "", "(株)": "", "㈱": "",
                   "有限会社": "", "（有）": "", "(有)": "",
                  }
    if type(x) is pd.core.series.Series:
        x = x.replace(remove_dict, regex=True)
    else:
        x = x.rename(index=remove_dict, regex=True)
    return x

In [36]:
# In[6]:
tenki_candidate['CompanyName'] = tenki_candidate['CompanyName'].apply(lambda x: jaconv.z2h(x))

In [37]:
tenki_candidate['変換CompanyName'] = remove_kabu(tenki_candidate['CompanyName'])

In [38]:
tenki_candidate['アンタッチャ'] = tenki_candidate['変換CompanyName'].apply(lambda x: get_untauch(x))

In [39]:
tenki_candidate = tenki_candidate.dropna(subset={'アンタッチャ'})

In [40]:
tenki_candidate = tenki_candidate[['tenki_ID','アンタッチャ','変換CompanyName']]

In [41]:
tenki_candidate

,tenki_ID,アンタッチャ,変換CompanyName
10,49926,現S同意書回収,歯愛ﾒﾃﾞｨｶﾙ
13,49929,現S同意書回収,LIXIL
14,49930,現S同意書回収,なし
15,49931,現S同意書回収,大京
17,49933,現S同意書回収,ﾊﾟﾅｿﾆｯｸ
...,...,...,...
44435,105329,現S同意書回収,日本電気
44440,105334,現S同意書回収,TOTO
44441,105335,現S同意書回収,ｵｰｼﾞｯｸ
44442,105336,現S同意書回収,ﾎﾟｰﾗ


In [42]:
#マスタの最後のIDと行番号を取得する
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '15qRR-yfJxgCh_TpXwBjBHNAnc8yAeTUCF0-Y_N6GoIo'
Sheet_NAME = '現Sチェック!A'
Sheet_row = ":A"
RANGE_NAME = Sheet_NAME+Sheet_row
recentID = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)
recentID

,ID
1,49926
2,49929
3,49930
4,49931
5,49933
...,...
9018,105291
9019,105298
9020,105303
9021,105314


In [43]:
#マスタの最後のID
cellnr = len(recentID)
latestID = recentID.at[cellnr, "ID"]
latestID = int(recentID['ID'][cellnr])
latestID

105315

In [24]:
tenki_candidate = tenki_candidate.query('tenki_ID >='+latestID)

,tenki_ID,アンタッチャ
10,49926,現S同意書回収
13,49929,現S同意書回収
14,49930,現S同意書回収
15,49931,現S同意書回収
17,49933,現S同意書回収
...,...,...
44429,105323,現S同意書回収
44435,105329,現S同意書回収
44440,105334,現S同意書回収
44441,105335,現S同意書回収


In [44]:
# In[8]:
tenki_untouch = tenki_candidate[['tenki_ID','アンタッチャ','変換CompanyName']].values.tolist()
tenki_untouch

[[49926, '現S同意書回収', '歯愛ﾒﾃﾞｨｶﾙ'],
 [49929, '現S同意書回収', 'LIXIL'],
 [49930, '現S同意書回収', 'なし'],
 [49931, '現S同意書回収', '大京'],
 [49933, '現S同意書回収', 'ﾊﾟﾅｿﾆｯｸ'],
 [49951, '現S同意書回収', 'ﾏﾂﾀﾞ'],
 [49955, '現S同意書回収', '博報堂'],
 [49957, '現S同意書回収', 'ﾐﾛｸ情報ｻｰﾋﾞｽ'],
 [49964, '現S同意書回収', 'ｷｰｴﾝｽ'],
 [49970, '現S同意書回収', 'ﾈｵｷｬﾘｱ'],
 [49974, '現S同意書回収', '富士薬品'],
 [49979, '現S同意書回収', 'TIS'],
 [49981, '現S同意書回収', '限責任あずさ監査法人'],
 [49986, '現S同意書回収', 'ｴｲｺｰ'],
 [49987, '現S同意書回収', '春'],
 [49990, '現S同意書回収', 'ﾆﾁﾚｲﾌｰｽﾞ'],
 [49993, '現S同意書回収', 'ﾃｸﾉ'],
 [49996, '現S同意書回収', 'ﾐｽﾐ'],
 [50000, '現S同意書回収', '味の素'],
 [50001, '現S同意書回収', 'ﾅｶﾞｵｶ'],
 [50003, '現S同意書回収', 'KNDｺｰﾎﾟﾚｰｼｮﾝ'],
 [50014, '現S同意書回収', 'ﾊﾟｲｵﾆｱ'],
 [50016, '現S同意書回収', 'ｵｰｹｰ'],
 [50019, '現S同意書回収', 'ﾎｰﾌﾟ'],
 [50032, '現S同意書回収', '加藤製作所'],
 [50034, '現S同意書回収', '文英堂'],
 [50037, '現S同意書回収', 'CITICCapitalPartners'],
 [50050, '現S同意書回収', '住友不動産'],
 [50058, '現S同意書回収', 'ﾗｲｵﾝ'],
 [50059, '現S同意書回収', 'ｼｰｼｰｴｽ'],
 [50061, '現S同意書回収', 'ﾒｯｸｲﾝﾀｰﾅｼｮﾅﾙ'],
 [50063, '現S同意書回収', 'なし'],
 [50065, '現S同意書回収', '

In [45]:
###ここを編集
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
SPREADSHEET_ID = '15qRR-yfJxgCh_TpXwBjBHNAnc8yAeTUCF0-Y_N6GoIo'
RANGE_NAME = '現Sチェック!A2'
json_path = "credentials.json"
service = ps.get_auth(SCOPES, json_path)
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,tenki_untouch,service)